Let's go through your code carefully.
```python
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=17)
c_values = np.logspace(-2, 3, 500)
logit_searcher = LogisticRegressionCV(
    Cs=c_values,
    cv=skf,
    n_jobs=-1
)
logit_searcher.fit(X_poly, y)
```

## What happens during `.fit()`?

`LogisticRegressionCV`:

1. Takes the 500 candidate values of $C$:
   ```python
   c_values = np.logspace(-2, 3, 500)
   ```
   which range roughly from
   $$10^{-2}=0.01$$
   to
   $$10^3=1000$$
2. For each $C$:
   * Performs 5-fold cross-validation.
   * Trains logistic regression on 4 folds.
   * Evaluates on the remaining fold.
   * Repeats for all 5 folds.
3. Computes the average score for each $C$.
4. Selects the best $C$.
5. Retrains a final logistic regression model on the entire dataset using that best $C$.

---
# What does `logit_searcher` contain?

After fitting, it contains:

### Learned coefficients
```python
logit_searcher.coef_
```
The learned weights $w$.

Example:
```python
array([[1.2, -0.7, 0.5]])
```

---
### Intercept
```python
logit_searcher.intercept_
```
The bias term.

---
### Candidate C values
```python
logit_searcher.Cs_
```
These are the 500 values generated by:
```python
np.logspace(-2, 3, 500)
```

---
### Best C
```python
logit_searcher.C_
```
Example:
```python
array([154.30384692])
```

Meaning:
> Among the 500 tested values, $C \approx 154.3$ achieved the highest average cross-validation score.

---
### Cross-validation scores
```python
logit_searcher.scores_
```
This stores all CV results.

---
# What is `logit_searcher.scores_[1]`?

For binary classification:
```python
logit_searcher.scores_
```
is usually a dictionary:
```python
{
    0: ...,
    1: ...
}
```
or sometimes only:
```python
{
    1: ...
}
```
depending on the sklearn version.

The key corresponds to the class label.

So:
```python
logit_searcher.scores_[1]
```
returns a matrix of CV scores.

Shape:
```python
(n_folds, n_C_values)
```

In your case:
```python
(5, 500)
```

because:
* 5 folds
* 500 candidate C values

---
Conceptually:
```python
scores_[1]
=
[
 fold1_scores,
 fold2_scores,
 fold3_scores,
 fold4_scores,
 fold5_scores
]
```

where each row contains the accuracy obtained for every $C$.

Example (using only 4 C values):
```python
[
 [0.80, 0.82, 0.84, 0.83],
 [0.81, 0.83, 0.85, 0.84],
 [0.79, 0.82, 0.86, 0.85],
 [0.80, 0.84, 0.87, 0.86],
 [0.81, 0.83, 0.86, 0.85]
]
```

Rows = folds
Columns = different C values

---
# Why do we compute the mean?

This line:
```python
np.mean(logit_searcher.scores_[1], axis=0)
```
averages across folds.

Result:
```python
[
 0.802,
 0.828,
 0.856,
 0.846
]
```

Now you have:

| C  | Mean CV Accuracy |
| -- | ---------------- |
| C₁ | 0.802            |
| C₂ | 0.828            |
| C₃ | 0.856            |
| C₄ | 0.846            |

---
# What is being plotted?

```python
plt.plot(
    c_values,
    np.mean(logit_searcher.scores_[1], axis=0)
)
```

plots:

* x-axis = candidate $C$ values
* y-axis = average CV accuracy

Conceptually:
```text
Accuracy
 ^
 |
 |                 *
 |               *   *
 |            *        *
 |         *
 |      *
 +---------------------------> C
```

The highest point corresponds to the selected:
```python
logit_searcher.C_
```

which in your case is:
```python
array([154.30384692])
```

So sklearn looked at all 500 columns in `scores_[1]`, found which column had the highest average CV score, and chose the corresponding $C$ value (≈154.3).